# Is the measured improvement resolved?

Chapter 7 · Day 10

Aligned item scores preserve the shared difficulty of a question. Explore how uncertainty differs from confidence in an individual answer.

Use the cycle: prediction → inspect mechanism → change one choice → interpret limits. Runnable reference answers follow each question. All computations use CPU; no model download or server is started.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
sys.path.insert(0, str(root / "src"))
import math, copy, json
import numpy as np
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "font.family": "DejaVu Sans", "font.size": 11})
from dongxi_llms import evaluation_lab as ev
from dongxi_llms import instruction_data_lab as data
from dongxi_llms import sft_lab as sft
def show_flow(labels, focus):
    fig, ax = plt.subplots(figsize=(11, 1.9))
    for i, label in enumerate(labels):
        ax.text(i, .5, label, ha="center", va="center",
                bbox=dict(boxstyle="round,pad=.45", fc="#dce9f5" if i == focus else "#f6f6f6", ec="#666"))
        if i: ax.annotate("", (i-.25,.5), (i-.75,.5), arrowprops=dict(arrowstyle="->",color="#555"))
    ax.set_xlim(-.6,len(labels)-.4); ax.set_ylim(0,1); ax.axis("off")
    ax.set_title("Schematic process: highlighted component is this lesson's focus")
    plt.show()


In [ ]:
show_flow(["Aligned items","Paired scores","Resample","Interval"], 2)

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![02 paired uncertainty: reference plot 01](../figures/chapter-07/day-10-02_paired_uncertainty-01.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

## Predict 1

Predict whether 10/10 correct proves a perfect success rate on future tasks. What changes when the independent sample size becomes 100?

Pause and explain your reasoning before running the next cell.

### Reference answer and mechanism

Both observed rates are one, but their intervals differ. Zero observed failures leaves uncertainty; more independent items narrow it.

In [ ]:
sizes = np.array([10,20,50,100,300])
intervals = np.array([ev.wilson_interval(int(n),int(n)) for n in sizes])
fig,ax = plt.subplots(figsize=(7,3.5))
ax.errorbar(sizes,np.ones(len(sizes)),yerr=[np.maximum(1-intervals[:,0],0),np.maximum(intervals[:,1]-1,0)],fmt="o",color="#346c9c",capsize=4)
ax.set(xlabel="Independent evaluation items",ylabel="Success rate / nominal 95% score interval",ylim=(.65,1.03))
plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![02 paired uncertainty: reference plot 02](../figures/chapter-07/day-10-02_paired_uncertainty-02.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

## Predict 2

The same forty items are evaluated by two fixture models. Predict why we should resample item pairs instead of independently mixing their score arrays.

Pause and explain your reasoning before running the next cell.

### Reference answer and mechanism

The paired difference preserves shared item difficulty. Each draw asks how the comparison would change under another sample of these items.

In [ ]:
rows = ev.evaluation_fixture()
a,b = [int(r["a"]) for r in rows],[int(r["b"]) for r in rows]
delta, interval = ev.paired_bootstrap(a,b,draws=2000)
rng = np.random.default_rng(1010)
differences = np.array(b)-np.array(a)
samples = differences[rng.integers(0,len(rows),size=(2000,len(rows)))].mean(1)
fig,ax = plt.subplots(figsize=(7,3.5))
ax.hist(samples,bins=18,color="#90b4d1",edgecolor="white")
ax.axvline(0,color="#555",linestyle="--"); ax.axvline(delta,color="#b16b31")
ax.set(xlabel="Resampled mean B − A",ylabel="Bootstrap draws",title="Paired uncertainty: fixture scores")
plt.show()
print("Observed difference and percentile interval:",delta,interval)

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![02 paired uncertainty: reference plot 03](../figures/chapter-07/day-10-02_paired_uncertainty-03.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

## Predict 3

What happens if every item is repeated ten times and treated as independent? Does the narrower interval represent ten times more knowledge?

Pause and explain your reasoning before running the next cell.

### Reference answer and mechanism

Duplicate items falsely increase the apparent independent sample count. The population information has not increased. Resample original groups rather than expanded rows.

In [ ]:
original = ev.wilson_interval(sum(a),len(a))
duplicated = ev.wilson_interval(10*sum(a),10*len(a))
print("Original interval:",original,"naive duplicate interval:",duplicated)
assert duplicated[1]-duplicated[0] < original[1]-original[0]

## Reading the evidence

The figures are regenerated from the current lesson tensors or declared fixture. Axes and counts define the comparison. Change one control, rerun the relevant cell, and explain what changed in the mechanism. Separate a verified implementation property from a claim about a real language model.

Return to the corresponding chapter and worked solution for the complete argument.